# PoWR + ATLAS/Phoenix fitting to Gaia BP/RP spectra
# v8.2 – With Model Manifest Support

**Changes from v8.1:**
- Loads models from `model_manifest.csv` if available (includes Mass, logL, R)
- Falls back to glob-based loading if no manifest
- Output CSV now includes: model_source, logL, Mass, Mass_std, R_Rsun
- Gordon+ 2023 extinction law (state-of-the-art)

**Input:**
- `model_manifest.csv` (from download notebook) OR glob model directories
- Catalog FITS file (columns: source_id, Gmag, parallax, parallax_error)
- `./BPRP_spectra/<source_id>.fits` (XP_SAMPLED format)

**Output:**
- CSV with: source_id, Teff, logg, A_V, R_V, distance_pc, gmag, M_G, chi2_red, 
  model_source, logL, Mass, Mass_std, R_Rsun
- `fit_plots/fit_<source_id>.png`

In [2]:
!pip install dust_extinction

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
import pandas as pd
import glob
import re
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

# Check for dust_extinction package
try:
    from dust_extinction.parameter_averages import G23
    # G24 is only in very recent versions; G23 is perfectly fine
    try:
        from dust_extinction.parameter_averages import G24
    except ImportError:
        G24 = None  # Not available, will use G23
    print("✓ dust_extinction package loaded successfully")
except ImportError:
    raise ImportError(
        "Please install dust_extinction >= 1.2:\n"
        "    pip install dust_extinction\n"
        "This package contains the official Gordon+ 2023 curves."
    )

✓ dust_extinction package loaded successfully


In [ ]:
# ========================== CONFIGURATION ==========================
POWR_MODEL_DIR   = Path('./griddl-gal-ob-vd3-line_calib')
ATLAS_MODEL_DIR  = Path('./stellar_models')
MODEL_MANIFEST   = Path('./model_manifest.csv')  # NEW: manifest with Mass, L, R
CATALOG_FITS     = 'JMH.SDSSV_massive_subsample.fits'
SPECTRA_DIR      = Path('./BPRP_spectra')
PLOT_DIR         = Path('./fit_plots')
PLOT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV       = 'Zari_G_bright_fits.csv'

# Fitting grid
A_V_GRID  = np.arange(0.0, 7.1, 0.1)
R_V_GRID  = np.array([2.5, 3.1, 3.7])
WAVELENGTH_FIT_MIN = 340.0   # nm
WAVELENGTH_FIT_MAX = 900.0   # nm
SYSTEMATIC_FLOOR   = 0.03
BLUE_WEIGHT_REGION = (340.0, 480.0)
BLUE_WEIGHT_FACTOR = 2.0

W_M2_NM_TO_ERG_S_CM2_A = 1e2   # unit conversion

# Check if manifest exists
USE_MANIFEST = MODEL_MANIFEST.exists()
if USE_MANIFEST:
    print(f"✓ Model manifest found: {MODEL_MANIFEST}")
else:
    print(f"⚠ No manifest at {MODEL_MANIFEST} - will use glob-based loading")

In [7]:
# ========================== LOAD CATALOG ==========================
print("Loading catalog...")
with fits.open(CATALOG_FITS) as hdul:
    data = hdul[1].data
    source_ids       = data['source_id']
    gmag             = data['Gmag']
    parallax         = data['parallax']
    parallax_error   = data['parallax_error']

names = [str(sid) for sid in source_ids]
print(f"→ {len(source_ids)} sources loaded")

Loading catalog...
→ 1679 sources loaded


In [ ]:
# ============================================================================
# LOAD ALL MODELS (from manifest if available, else glob)
# ============================================================================

all_models = []  # Will hold all models with their properties

if USE_MANIFEST:
    # ========== NEW: Load from model_manifest.csv ==========
    print(f"\nLoading models from manifest: {MODEL_MANIFEST}")
    
    manifest_df = pd.read_csv(MODEL_MANIFEST)
    print(f"  Manifest contains {len(manifest_df)} models")
    print(f"  Columns: {list(manifest_df.columns)}")
    
    # Show coverage
    for src in manifest_df['source'].unique():
        subset = manifest_df[manifest_df['source'] == src]
        print(f"  {src}: {len(subset)} models, Teff={subset['Teff'].min()}-{subset['Teff'].max()}K")
    
    # Load each model's spectrum
    for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Loading models"):
        source = row['source']
        filename = row['filename']
        
        # Determine file path based on source
        if source == 'PoWR':
            filepath = POWR_MODEL_DIR / filename
        else:  # ATLAS or PHOENIX
            filepath = ATLAS_MODEL_DIR / filename
        
        if not filepath.exists():
            print(f"  Warning: {filepath} not found, skipping")
            continue
        
        try:
            data = np.loadtxt(filepath)
            wave_model = data[:, 0]  # Angstroms
            log_flux_model = data[:, 1]
            flux_model = 10**log_flux_model  # erg/s/cm²/Å at 10 pc
            
            all_models.append({
                'Teff': row['Teff'],
                'logg': row['logg'],
                'source': source,
                'logL': row['logL'],
                'Mass': row['Mass'],
                'Mass_std': row.get('Mass_std', np.nan),
                'R_Rsun': row['R_Rsun'],
                'wavelength': wave_model,
                'flux': flux_model,
                'filename': filename
            })
        except Exception as e:
            print(f"  Error loading {filepath}: {e}")
    
    print(f"\n✓ Successfully loaded {len(all_models)} models from manifest")

else:
    # ========== FALLBACK: Original glob-based loading ==========
    print("\nLoading PoWR models (glob mode)...")
    
    model_files = sorted(glob.glob(str(POWR_MODEL_DIR / 'gal-ob-vd3_*_line_calib.txt')))
    print(f"  Found {len(model_files)} PoWR model files")
    
    for model_file in tqdm(model_files, desc="Loading PoWR models"):
        match = re.search(r'gal-ob-vd3_(\d+)-(\d+)_line_calib\.txt', model_file)
        if not match:
            continue
        
        Teff = int(match.group(1)) * 1000
        logg = int(match.group(2)) / 10.0
        
        try:
            data = np.loadtxt(model_file)
            wave_model = data[:, 0]
            log_flux_model = data[:, 1]
            flux_model = 10**log_flux_model
            
            all_models.append({
                'Teff': Teff,
                'logg': logg,
                'source': 'PoWR',
                'logL': np.nan,  # Not available without manifest
                'Mass': np.nan,
                'Mass_std': np.nan,
                'R_Rsun': np.nan,
                'wavelength': wave_model,
                'flux': flux_model,
                'filename': Path(model_file).name
            })
        except Exception as e:
            print(f"  Error loading {model_file}: {e}")
    
    print(f"  Loaded {len(all_models)} PoWR models")
    
    # Load ATLAS/PHOENIX models
    print("\nLoading ATLAS/PHOENIX models (glob mode)...")
    
    atlas_files = sorted(glob.glob(str(ATLAS_MODEL_DIR / 'atlas*.txt')))
    print(f"  Found {len(atlas_files)} ATLAS/PHOENIX model files")
    
    pattern = re.compile(r'(\d{2,3})-(\d{2})(?:\.txt)?$')
    
    for model_file in tqdm(atlas_files, desc="Loading ATLAS/PHOENIX models"):
        m = pattern.search(Path(model_file).name)
        if not m:
            continue
        
        teff_code = m.group(1)
        logg_code = m.group(2)
        
        try:
            logg = int(logg_code) / 10.0
            if len(teff_code) == 3:
                Teff = int(teff_code) * 100
            elif len(teff_code) == 2:
                v = int(teff_code)
                Teff = v * 1000 if v <= 20 else v * 100
            else:
                Teff = int(teff_code) * 100
        except ValueError:
            continue
        
        try:
            data = np.loadtxt(model_file)
            wave_model = data[:, 0]
            log_flux_model = data[:, 1]
            flux_model = 10**log_flux_model
            
            all_models.append({
                'Teff': Teff,
                'logg': logg,
                'source': 'ATLAS',
                'logL': np.nan,
                'Mass': np.nan,
                'Mass_std': np.nan,
                'R_Rsun': np.nan,
                'wavelength': wave_model,
                'flux': flux_model,
                'filename': Path(model_file).name
            })
        except Exception as e:
            print(f"  Error loading {model_file}: {e}")
    
    print(f"\n✓ Total models loaded (glob mode): {len(all_models)}")
    print("  Note: Mass, logL, R not available without manifest")

# Rename for compatibility with rest of notebook
powr_models = all_models
print(f"\nModel library ready: {len(powr_models)} models")

In [9]:
# ========================== GAIA GRID & RESAMPLE ==========================
# Take first spectrum to get the exact Gaia wavelength grid
ref_file = next(SPECTRA_DIR.glob('*.fits'))
with fits.open(ref_file) as hdul:
    gaia_wavelength_nm = np.asarray(hdul[1].data['wavelength']).flatten()
gaia_wavelength_A = gaia_wavelength_nm * 10

# Build bin edges
diff = np.diff(gaia_wavelength_A)
bins = np.concatenate([[gaia_wavelength_A[0]-diff[0]/2],
                       gaia_wavelength_A[:-1] + diff/2,
                       [gaia_wavelength_A[-1]+diff[-1]/2]])

# Resample every model
print("Resampling models to Gaia grid...")
for m in tqdm(powr_models):
    flux = np.zeros(len(gaia_wavelength_A))
    for i in range(len(gaia_wavelength_A)):
        mask = (m['wavelength'] >= bins[i]) & (m['wavelength'] < bins[i+1])
        if mask.sum()>0:
            flux[i] = np.mean(m['flux'][mask])
        else:
            flux[i] = np.interp(gaia_wavelength_A[i], m['wavelength'], m['flux'])
    flux = gaussian_filter1d(flux, sigma=2.0)   # same smoothing as before
    m['flux_gaia'] = flux
    m['wave_gaia'] = gaia_wavelength_A

Resampling models to Gaia grid...


100%|█████████████████████████████████████████████████████████████| 402/402 [01:46<00:00,  3.78it/s]


In [10]:
# ============================================================================
# EXTINCTION FUNCTION – GORDON+ 2024 (state-of-the-art, used in Zhang+ 2025)
# ============================================================================
# This replaces CCM89 with the modern Gordon et al. (2024) R(V)-dependent curves.
# Reference: Gordon et al. 2023, ApJ, 950, 86 (G23)
#            Updated coefficients in G24 (2024)
# Used in Zhang et al. 2025 (Science), Weiler 2024, Maíz Apellániz+ 2024, etc.
#
# The G23/G24 models provide:
#   A(λ)/A(V) as a function of wavelength and R(V)
# Valid for R(V) = 2.3–5.6, wavelength = 912 Å – 32 μm
# ============================================================================

import astropy.units as u

# Choose extinction model - G23 is well-tested and widely used
# G24 has minor updates but may not be available in older versions
if G24 is not None:
    ExtinctionModel = G24
    print("Using extinction model: G24 (Gordon+ 2024)")
else:
    ExtinctionModel = G23
    print("Using extinction model: G23 (Gordon+ 2023)")

# ============================================================================
# Pre-compute extinction lookup table for speed (recommended)
# This is ~10–20× faster in the inner fitting loop
# ============================================================================

def build_extinction_table(wavelength_aa, R_V_grid):
    """
    Pre-compute A(λ)/A_V for all R_V values on the exact Gaia wavelength grid.
    Returns dict: R_V → A(λ)/A_V array
    
    Parameters
    ----------
    wavelength_aa : array [Å]
        Wavelength grid in Angstroms
    R_V_grid : array
        R_V values to pre-compute
    
    Returns
    -------
    table : dict
        Dictionary mapping R_V → A(λ)/A_V array
    """
    table = {}
    # dust_extinction expects wavelength with units
    wave_with_units = wavelength_aa * u.AA
    
    for Rv in R_V_grid:
        ext = ExtinctionModel(Rv=Rv)
        # ext(wavelength) returns A(λ)/A(V)
        table[Rv] = ext(wave_with_units)
    return table

# Build extinction table at startup
print(f"Building {ExtinctionModel.__name__} extinction lookup table...")
EXT_TABLE = build_extinction_table(gaia_wavelength_A, R_V_GRID)
print(f"   → Extinction table ready for R_V = {list(R_V_GRID)}")

# ============================================================================
# Reddening functions
# ============================================================================

def apply_reddening(wavelength_aa, flux, A_V, R_V=3.1):
    """
    Apply Gordon+ 2023/2024 extinction to a spectrum.
    Uses pre-computed lookup table for speed.
    
    Parameters
    ----------
    wavelength_aa : array [Å]
        Wavelength in Angstroms (not used if EXT_TABLE exists on same grid)
    flux : array
        Intrinsic flux (any units, must match wavelength grid)
    A_V : float
        Visual extinction in magnitudes
    R_V : float
        Selective extinction (must be in R_V_GRID for fast lookup)
    
    Returns
    -------
    reddened_flux : array
        Extincted flux
    """
    # Use pre-computed table if R_V is in the grid
    if R_V in EXT_TABLE:
        A_lambda_over_Av = EXT_TABLE[R_V]
    else:
        # Fall back to computing on the fly
        ext = ExtinctionModel(Rv=R_V)
        A_lambda_over_Av = ext(wavelength_aa * u.AA)
    
    A_lambda = A_lambda_over_Av * A_V
    reddened_flux = flux * 10**(-0.4 * A_lambda)
    return reddened_flux


def calculate_A_G(A_V, R_V=3.1):
    """
    Compute A_G (extinction in Gaia G band) from A_V.
    Uses Gordon+ 2023/2024 extinction law at the effective wavelength
    of the Gaia G band (~6420 Å for DR3).
    
    Parameters
    ----------
    A_V : float
        V-band extinction in magnitudes
    R_V : float
        Selective extinction ratio
    
    Returns
    -------
    A_G : float
        Gaia G-band extinction in magnitudes
    """
    lambda_G_aa = 6420.0  # Effective wavelength for Gaia DR3 G band
    ext = ExtinctionModel(Rv=R_V)
    A_lambda_over_Av = ext(lambda_G_aa * u.AA)
    return float(A_lambda_over_Av) * A_V


# ============================================================================
# Wavelength weights for chi-square fitting
# ============================================================================

def wavelength_weights(wavelength_nm, blue_region=BLUE_WEIGHT_REGION, blue_weight=BLUE_WEIGHT_FACTOR):
    """
    Return weights for chi-square as function of wavelength.
    Gives extra weight to blue region where extinction effects are stronger.
    
    Parameters
    ----------
    wavelength_nm : array
        Wavelengths in nm
    blue_region : tuple
        (lambda_min, lambda_max) in nm for enhanced weighting
    blue_weight : float
        Weight factor for blue region (1.0 = equal weight everywhere)
    
    Returns
    -------
    weights : array
        Weights for each wavelength point
    """
    weights = np.ones_like(wavelength_nm)
    
    # Apply enhanced weight to blue region
    blue_mask = (wavelength_nm >= blue_region[0]) & (wavelength_nm <= blue_region[1])
    weights[blue_mask] = blue_weight
    
    return weights

Using extinction model: G23 (Gordon+ 2023)
Building G23 extinction lookup table...
   → Extinction table ready for R_V = [np.float64(2.5), np.float64(3.1), np.float64(3.7)]


In [11]:
# ========================== FITTING FUNCTION ==========================
def fit_single_star(sid, plx, plx_err, gmag):
    file = SPECTRA_DIR / f"{sid}.fits"
    if not file.exists():
        return None
    
    with fits.open(file) as h:
        d = h[1].data
        w_nm = d['wavelength'].flatten()
        f    = d['flux'].flatten() * 1e2
        err  = d['flux_error'].flatten() * 1e2 if 'flux_error' in d.names else f*0.01
    
    err = np.sqrt(err**2 + (0.03*f)**2)
    w_A = w_nm * 10
    mask = (w_A >= 3400) & (w_A <= 9000)
    if mask.sum() < 10:
        return None
    
    # Helper function to evaluate chi2 for all models and Rv at a given Av
    def evaluate_av(av_val):
        """Find best chi2 across all models and Rv values for a given Av"""
        best_chi2_for_av = np.inf
        best_for_av = None
        
        for m in powr_models:
            for Rv in R_V_GRID:
                fmod = apply_reddening(m['wave_gaia'], m['flux_gaia'], av_val, Rv)
                scale = np.sum(f[mask] * fmod[mask] / err[mask]**2) / np.sum(fmod[mask]**2 / err[mask]**2)
                dist = 10.0 / np.sqrt(scale)
                chi_spec = np.sum(wavelength_weights(w_nm[mask]) * ((f[mask] - scale*fmod[mask]) / err[mask])**2)
                chi_plx  = ((1000/dist - plx) / plx_err)**2 if (plx > 0 and plx_err > 0 and np.isfinite(plx_err)) else 0
                chi_tot  = chi_spec + chi_plx
                
                if chi_tot < best_chi2_for_av:
                    best_chi2_for_av = chi_tot
                    best_for_av = {**m, 'A_V':av_val, 'R_V':Rv, 'scale':scale, 'dist_pc':dist,
                                   'chi2_tot':chi_tot, 'chi2_spec':chi_spec, 'chi2_plx':chi_plx,
                                   'flux_mod':fmod*scale, 'w_obs':w_nm, 'f_obs':f, 'err_obs':err}
        
        return best_chi2_for_av, best_for_av
    
    # Stage 1: Coarse grid search with more points for robustness
    av_coarse = np.arange(0, 6.05, 0.5)  # [0.0, 0.5, 1.0, 1.5, ...]
    
    chi2_coarse = []
    results_coarse = []
    
    for av_val in av_coarse:
        chi2, result = evaluate_av(av_val)
        chi2_coarse.append(chi2)
        results_coarse.append(result)
    
    chi2_coarse = np.array(chi2_coarse)
    best_coarse_idx = np.argmin(chi2_coarse)
    
    # Stage 2: Refine around the best point with iterative grid refinement
    # Create a refined grid centered on the best coarse point
    av_center = av_coarse[best_coarse_idx]
    search_radius = 0.5
    
    for iteration in range(3):
        # Create refined grid around center
        av_refined = np.linspace(max(0, av_center - search_radius), 
                                 min(6.0, av_center + search_radius), 
                                 5)
        
        chi2_refined = []
        results_refined = []
        
        for av_val in av_refined:
            chi2, result = evaluate_av(av_val)
            chi2_refined.append(chi2)
            results_refined.append(result)
        
        chi2_refined = np.array(chi2_refined)
        best_refined_idx = np.argmin(chi2_refined)
        
        # Try parabolic fit if we have a good interior minimum
        if 0 < best_refined_idx < len(av_refined) - 1:
            # Use three points around minimum for parabola
            idx_range = [best_refined_idx - 1, best_refined_idx, best_refined_idx + 1]
            av_pts = av_refined[idx_range]
            chi_pts = chi2_refined[idx_range]
            
            # Fit parabola
            try:
                coeffs = np.polyfit(av_pts, chi_pts, 2)
                a, b, c = coeffs
                
                # Check if parabola is valid (opens upward and minimum is reasonable)
                if a > 0:
                    av_parabolic = -b / (2 * a)
                    # Only use parabolic minimum if it's within our search range
                    if av_pts[0] <= av_parabolic <= av_pts[2]:
                        av_center = av_parabolic
                    else:
                        av_center = av_refined[best_refined_idx]
                else:
                    av_center = av_refined[best_refined_idx]
            except:
                # If parabolic fit fails, just use grid minimum
                av_center = av_refined[best_refined_idx]
        else:
            # Minimum is at boundary, use that point
            av_center = av_refined[best_refined_idx]
        
        # Clamp to valid range
        av_center = np.clip(av_center, 0, 6.0)
        
        # Reduce search radius for next iteration
        search_radius = search_radius / 2.5
    
    # Stage 3: Final evaluation at the refined center
    chi2_final, best_result = evaluate_av(av_center)
    
    # Safety check: compare with best from coarse grid
    if chi2_coarse[best_coarse_idx] < chi2_final:
        # If coarse grid was actually better, use that (shouldn't happen but be safe)
        return results_coarse[best_coarse_idx]
    
    return best_result

In [12]:
# ========================== PLOTTING FUNCTION (FIXED) ==========================
PLOT_DIR = Path('./fit_plots')
PLOT_DIR.mkdir(exist_ok=True)

def plot_fit(sid, fit_params):
    """
    Safe plotting function – works even if distance is nan or mask is missing.
    """
    if fit_params is None:
        return

    w_nm  = fit_params['w_obs']           # full observed wavelength grid (nm)
    f_obs = fit_params['f_obs']
    f_mod = fit_params['flux_mod']
    err   = fit_params['err_obs']

    # Re-create the exact same mask that was used during fitting
    w_A   = w_nm * 10.0
    mask  = (w_A >= 3400) & (w_A <= 9000)

    # --- Distance handling (protect against nan) ---
    dist_pc = fit_params['dist_pc']
    if not np.isfinite(dist_pc) or dist_pc <= 0:
        dist_pc = np.nan
        dist_str = "???"
    else:
        dist_str = f"{dist_pc:.0f}"

    # --- χ²_red calculation (robust) ---
    n_spec = mask.sum()
    n_plx  = 1 if fit_params.get('chi2_plx', 0) > 0 else 0
    dof    = n_spec + n_plx - 1
    chi2_red = fit_params['chi2_tot'] / dof if dof > 0 else 999.99

    # --------------------- Plot ---------------------
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                   gridspec_kw={'height_ratios': [3, 1]})

    # Top panel – spectrum
    ax1.plot(w_nm, f_obs, 'ko', ms=3, alpha=0.7, label='Gaia BP/RP')
    ax1.plot(w_nm, f_mod, 'r-', lw=2,
             label=f"Teff={fit_params['Teff']/1000:.0f} kK  log g={fit_params['logg']:.2f}\n"
                   f"A_V={fit_params['A_V']:.2f}  R_V={fit_params['R_V']:.1f}")

    # Rayleigh-Jeans reference tail
    w_ref = 600.0
    f_ref = np.interp(w_ref, w_nm, f_mod, left=np.nan, right=np.nan)
    if np.isfinite(f_ref):
        f_ref *= 0.9
        w_rj = np.linspace(340, 1050, 200)
        ax1.plot(w_rj, f_ref * (w_ref/w_rj)**4, '--', color='blue', lw=1.5,
                 alpha=0.8, label='Rayleigh-Jeans')

    ax1.set_yscale('log')
    ax1.set_xlim(330, 1050)
    ax1.set_ylabel('Flux (erg s⁻¹ cm⁻² Å⁻¹)')
    ax1.legend(fontsize=10)
    ax1.set_title(f"source_id = {sid} | d = {dist_str} pc | χ²_red = {chi2_red:.2f}")

    # Bottom panel – residuals (only in fitted region)
    resid = (f_obs - f_mod) / err
    ax2.plot(w_nm, resid, 'ko', ms=3, alpha=0.7)
    ax2.axhline(0, color='red', ls='--', lw=1)
    ax2.axhspan(-3, 3, color='gray', alpha=0.1)
    ax2.set_xlabel('Wavelength (nm)')
    ax2.set_ylabel('Residual (σ)')
    ax2.set_xlim(330, 1050)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    outfile = PLOT_DIR / f"fit_{sid}.png"
    plt.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.close()

    print(f" → plot saved: {outfile.name}")

In [ ]:
# ========================== MAIN LOOP ==========================
results = []
for i, sid in enumerate(source_ids,1):
    print(f"[{i:4d}/{len(source_ids)}] {sid} ", end='')
    res = fit_single_star(sid, parallax[i-1], parallax_error[i-1], gmag[i-1])
    if not res:
        print("no spectrum")
        continue
    A_G = calculate_A_G(res['A_V'], res['R_V'])
    M_G = gmag[i-1] - 5*np.log10(res['dist_pc']) + 5 - A_G
    
    # Build result dict with all parameters
    result_dict = {
        'source_id': sid,
        'Teff': res['Teff'],
        'logg': res['logg'],
        'A_V': res['A_V'],
        'R_V': res['R_V'],
        'distance_pc': res['dist_pc'],
        'gmag': gmag[i-1],
        'M_G': M_G,
        'chi2_red': res['chi2_tot']/(len(res['w_obs'])-1),
        # NEW: Include stellar properties from isochrones (via manifest)
        'model_source': res.get('source', 'unknown'),
        'logL': res.get('logL', np.nan),
        'Mass': res.get('Mass', np.nan),
        'Mass_std': res.get('Mass_std', np.nan),
        'R_Rsun': res.get('R_Rsun', np.nan),
    }
    results.append(result_dict)
    
    plot_fit(sid, res)
    
    # Print summary with Mass if available
    mass_str = f" M={res.get('Mass', np.nan):.1f}Msun" if np.isfinite(res.get('Mass', np.nan)) else ""
    print(f"Teff={res['Teff']/1000:.0f}kK logg={res['logg']:.1f} A_V={res['A_V']:.2f} d={res['dist_pc']:.0f}pc{mass_str}")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nFinished! → {OUTPUT_CSV}")
print(f"Output columns: {list(df.columns)}")